# 02 — M1 baseline (TV1, Việc 5 + Việc 6)

**Add Input:** Kaggle Dataset `nhom05-falldet-processed`. Bật **GPU** và **Internet** (tải trọng số YOLO11 lần đầu).

Luồng: tạo tracks chính thức (YOLO11m + ByteTrack) → kiểm tra chất lượng detector → tune trên **val** → chạy M1 → đánh giá trên **test** của URFD và Le2i → đánh giá cross-dataset.
Ngưỡng chỉ chọn trên val, không bao giờ dùng test.

In [ ]:
# Ô đầu tiên: lấy mã nguồn, cố định seed, in version Python, torch, ultralytics, opencv và version dataset.
# Cần bật Internet để clone repo.
import os, sys
BRANCH = "main"  # nhánh tv1/gk-data-eda-m1 đã merge, thiếu các bản sửa sau đó
REPO = "/kaggle/working/FallDetection"
if not os.path.exists(REPO):
    !git clone --depth 1 -b {BRANCH} https://github.com/ducqwang/FallDetection.git {REPO}
%cd {REPO}
!pip install -q ultralytics==8.4.202 lap==0.5.14
sys.path.insert(0, REPO)
from src.data.env_info import setup_notebook
setup_notebook("configs/data.yaml", seed=42)

In [ ]:
import json
from pathlib import Path
from src.data.loader import data_root
ROOT = data_root()
META = f"{ROOT}/metadata.csv"
GT = f"{ROOT}/annotations/le2i_gt_bbox.csv"
WORK = Path("/kaggle/working/m1")
WORK.mkdir(exist_ok=True)
# Tracks chính thức của TV1. Nếu đã có Kaggle Dataset nhom05-falldet-tracks (Add Input), trỏ vào đó
# và bỏ qua ô tạo tracks, ví dụ: TRACKS_DIR = Path("/kaggle/input/nhom05-falldet-tracks")
TRACKS_DIR = Path("/kaggle/working/tracks")
TRACKS = TRACKS_DIR / "tracks.parquet"
RESULTS = "reports/results"
print("Dữ liệu:", ROOT, "| Tracks:", TRACKS)

## 1. Tạo tracks chính thức (TV1) — cần GPU + Internet

`scripts/make_tracks.py`: YOLO11m (`yolo11m.pt`) + ByteTrack, chỉ lớp người, tracker reset giữa các video.
Mỗi video ghi một shard vào `tracks/shards/` ngay khi xong; Kaggle ngắt thì chạy lại đúng ô này, `--resume` (mặc định bật) bỏ qua video đã xong.
Shard nằm trong `/kaggle/working`, sẽ mất nếu phiên kết thúc mà không lưu: bật Persistence (Files) trong Notebook settings nếu có, hoặc tạo Dataset từ thư mục `tracks/` (mục 1c) rồi Add Input lại.
Ước lượng (chưa đo): khoảng 25–45 phút trên GPU T4/P100 cho 200 video (54 002 frame).

Script thoát mã khác 0 (và **không ghi** `tracks.parquet`) nếu có video lệch số frame quá 1 hoặc tracks vi phạm schema; ô kế tiếp sẽ dừng ở `assert`.

In [ ]:
!python scripts/make_tracks.py --root {ROOT} --out-dir {TRACKS_DIR} --dataset all --model yolo11m.pt

In [ ]:
# Sidecar: tham số, phiên bản mô hình/thư viện, git commit, số dòng đã bỏ, cảnh báo số frame lệch.
assert TRACKS.is_file(), "Chưa có tracks.parquet: xem dòng LỖI ở ô trên"
info = json.loads((TRACKS_DIR / "tracks_meta.json").read_text(encoding="utf-8"))
for key in ("model", "weights_sha256", "ultralytics", "torch", "opencv", "git_commit", "git_dirty", "args",
            "n_videos", "n_rows", "n_tracks", "dropped", "warnings", "errors", "videos_without_tracks",
            "validation", "run_seconds"):
    print(f"{key}: {info.get(key)}")
print("tracker yaml:\n" + (info.get("tracker_yaml") or ""))

### 1b. Chất lượng detector (tách lỗi detector khỏi lỗi luật M1)

Le2i: IoU trung bình của track chính so với GT và tỉ lệ frame GT có IoU ≥ 0.5, theo scene và theo lớp fall/ADL.
URFD: số track trung bình mỗi video và tỉ lệ frame có ít nhất một track. CSV ghi vào `tracks/quality/`.
Track chính Le2i được chọn bằng chính GT nên đây là cận trên của chất lượng tracks mà M1 nhận.

In [ ]:
!python scripts/check_tracks_quality.py --tracks {TRACKS} --root {ROOT}

### 1c. Lưu tracks thành Kaggle Dataset `nhom05-falldet-tracks` (private, chia sẻ nhóm)

Làm một lần sau khi ô 1 chạy xong, để M2/M3 và các lần chạy sau dùng **đúng cùng một** tracks.
1. Add-ons → Secrets: thêm `KAGGLE_USERNAME`, `KAGGLE_KEY` (từ kaggle.com → Settings → API → Create New Token), bật cho notebook này.
2. Chạy (đổi `<username>`; `kaggle datasets create` mặc định là **private**):
```python
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
(TRACKS_DIR / "dataset-metadata.json").write_text(json.dumps(
    {"title": "nhom05-falldet-tracks", "id": "<username>/nhom05-falldet-tracks", "licenses": [{"name": "other"}]}))
!kaggle datasets create -p {TRACKS_DIR} --dir-mode zip
# Lần sau (chạy lại tracks): !kaggle datasets version -p {TRACKS_DIR} --dir-mode zip -m "mô tả thay đổi"
```
3. Trên trang dataset: Settings → Sharing → thêm tài khoản các thành viên nhóm. Kiểm tra lại Visibility = Private.

Thư mục gồm `tracks.parquet`, `tracks_meta.json`, `quality/` và `shards/` (giữ shards để chạy tiếp được). Người dùng lại: Add Input `nhom05-falldet-tracks`, đặt `TRACKS_DIR` ở ô đầu, bỏ qua ô 1.

## 2. Tune trên tập val

In [ ]:
!python scripts/tune_m1.py --tracks {TRACKS} --meta {META} --dataset urfd --out {RESULTS}/m1_tuned_urfd.yaml
!python scripts/tune_m1.py --tracks {TRACKS} --meta {META} --dataset le2i --out {RESULTS}/m1_tuned_le2i.yaml
!python scripts/tune_m1.py --gt-bbox {GT} --meta {META} --dataset le2i --out {RESULTS}/m1_tuned_le2i_gt.yaml

## 3. Chạy M1 với ngưỡng đã tune

In [ ]:
!python -m src.models.m1_rule --tracks {TRACKS} --meta {META} --tuned {RESULTS}/m1_tuned_urfd.yaml --out {WORK}/m1_urfd.csv
!python -m src.models.m1_rule --tracks {TRACKS} --meta {META} --tuned {RESULTS}/m1_tuned_le2i.yaml --out {WORK}/m1_le2i.csv
!python -m src.models.m1_rule --gt-bbox {GT} --meta {META} --tuned {RESULTS}/m1_tuned_le2i_gt.yaml --out {WORK}/m1_le2i_gt.csv

## 4. Đánh giá trên tập test (T = 0.5, 1.0, 2.0)

Le2i GT là **kết quả cận trên**: so với dòng dùng tracks để tách lỗi của mô hình với lỗi của detector.
Mỗi bảng có mục ADL negative khó (`adl_hard_neg = 1`, chỉ có ở URFD).

In [ ]:
!python scripts/evaluate.py --pred {WORK}/m1_urfd.csv --meta {META} --split test --dataset urfd --tuned {RESULTS}/m1_tuned_urfd.yaml --T 0.5 1.0 2.0 --tag m1_tracks
!python scripts/evaluate.py --pred {WORK}/m1_le2i.csv --meta {META} --split test --dataset le2i --tuned {RESULTS}/m1_tuned_le2i.yaml --T 0.5 1.0 2.0 --tag m1_tracks
!python scripts/evaluate.py --pred {WORK}/m1_le2i_gt.csv --meta {META} --split test --dataset le2i --tuned {RESULTS}/m1_tuned_le2i_gt.yaml --T 0.5 1.0 2.0 --tag m1_gt

## 5. Cross-dataset

Le2i chỉ có khoảng 5 ADL trong tập test nên kết quả within-dataset nhiễu; ưu tiên báo cáo các dòng này.
Protocol A: ngưỡng tune trên URFD → toàn bộ 130 video Le2i. Protocol B: ngược lại → toàn bộ 70 video URFD.

In [ ]:
!python scripts/evaluate.py --pred {WORK}/m1_urfd.csv --meta {META} --split all --dataset le2i --tuned {RESULTS}/m1_tuned_urfd.yaml --tag m1_cross_A
!python scripts/evaluate.py --pred {WORK}/m1_le2i.csv --meta {META} --split all --dataset urfd --tuned {RESULTS}/m1_tuned_le2i.yaml --tag m1_cross_B

## 6. Bảng tổng hợp

In [ ]:
!python scripts/summarize_results.py --results-dir {RESULTS} --out {RESULTS}/m1_summary.csv

Tải `reports/results/*` về và commit lên nhánh.